In [18]:
# import env vars
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
## make a client
from anthropic import Anthropic
client = Anthropic()
model ="claude-sonnet-4-0"

In [20]:
## user func
def add_user_messages(messages,text):
    user_message = {"role":"user","content":text}
    messages.append(user_message)

## assistant func
def add_assistant_messages(messages,text):
    assistant_message = {"role":"assistant","content":text}
    messages.append(assistant_message)

## chat func
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [21]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, 
or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON,
 or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages=[]
    add_user_messages(messages,prompt)
    add_assistant_messages(messages,"```json")
    text=chat(messages,stop_sequences=["```"])
    return json.loads(text)

In [22]:
dataset=generate_dataset()
dataset

/var/folders/hl/xvs6dldn7nl02zjtlhj9vgbr0000gn/T/ipykernel_949/4216836428.py:24: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(**params)


[{'task': 'Write a Python function that takes an S3 bucket name and object key as parameters and returns the correct S3 object URL format (s3://bucket-name/object-key)'},
 {'task': "Create a JSON object that defines an IAM policy allowing read-only access to a specific S3 bucket named 'my-company-data'"},
 {'task': 'Write a regex pattern that validates AWS Lambda function names according to AWS naming conventions (1-64 characters, letters, numbers, hyphens, and underscores only, cannot start or end with hyphen)'}]

In [33]:
## Write the output in the json
with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

In [50]:
## func-1 
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""Please solve the following task:
    {test_case["task"]}
    """
    print("6..")
    messages=[]
    add_user_messages(messages,prompt)
    output=chat(messages)
    return output

In [51]:
## func-2
def run_test_case(test_case):
    """Calls run_prompt , then grades the result"""
    print("4...")
    output= run_prompt(test_case)
    print("5...")
    # TODO -Grading
    score = 10
    
    return {
        "output":output,
        "test_case": test_case,
        "score":score
    }

In [52]:
## func-3
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results=[]
    print("1-----")
    for test_case in dataset:
        print("2-----")
        result = run_test_case(test_case)
        print("3-----")
        results.append(result)

    return results    


In [55]:
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)   


1-----
2-----
4...
6..


/var/folders/hl/xvs6dldn7nl02zjtlhj9vgbr0000gn/T/ipykernel_949/4216836428.py:24: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(**params)


5...
3-----
2-----
4...
6..
5...
3-----
2-----
4...
6..
5...
3-----


In [56]:
print(json.dumps(results,indent=2))

[
  {
    "output": "Here's a Python function that creates the correct S3 object URL format:\n\n```python\ndef create_s3_url(bucket_name, object_key):\n    \"\"\"\n    Creates an S3 URL in the format s3://bucket-name/object-key\n    \n    Args:\n        bucket_name (str): The name of the S3 bucket\n        object_key (str): The key/path of the object in the bucket\n    \n    Returns:\n        str: The formatted S3 URL\n    \n    Raises:\n        ValueError: If bucket_name or object_key is empty or None\n    \"\"\"\n    # Validate inputs\n    if not bucket_name or not isinstance(bucket_name, str):\n        raise ValueError(\"Bucket name must be a non-empty string\")\n    \n    if not object_key or not isinstance(object_key, str):\n        raise ValueError(\"Object key must be a non-empty string\")\n    \n    # Remove leading slash from object_key if present\n    object_key = object_key.lstrip('/')\n    \n    # Create and return the S3 URL\n    return f\"s3://{bucket_name}/{object_key}\"